# AlphaForge — Leverage-Aware Training

**GPU Compute Provider: Google Colab**

This notebook runs the AlphaForge leverage-aware pipeline on GPU.
Steps: GPU check → clone repo → install deps → data → simulation → training.

In [ ]:
# ═══════════════════════════════════════════════════════════
# Step 1: GPU Check
# ═══════════════════════════════════════════════════════════
import sys, subprocess, json, os
from pathlib import Path

# Wand Colab çalışma dizini
NOTEBOOK_DIR = Path.cwd()
print(f"Notebook dir: {NOTEBOOK_DIR}")

# GPU doğrulama
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True, timeout=10,
)
if result.returncode == 0 and result.stdout.strip():
    print("✅ GPU DETECTED")
    print(result.stdout)
else:
    print("❌ NO GPU — Colab runtime > Change runtime type > T4 GPU")
    print("   Devam etmek için Runtime > Change runtime type > Hardware accelerator > GPU")


In [ ]:
# ═══════════════════════════════════════════════════════════
# Step 2: Clone / Pull Repo
# ═══════════════════════════════════════════════════════════
REPO_URL = "https://github.com/ddawnlll/v7-engine"
BRANCH = "main"

if Path("v7-engine").exists():
    !cd v7-engine && git pull origin {BRANCH}
    print("✅ Repo pulled")
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL}
    print("✅ Repo cloned")

%cd v7-engine
REPO_DIR = Path.cwd()
print(f"Working in: {REPO_DIR}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# Step 3: Install Dependencies
# ═══════════════════════════════════════════════════════════
print("Installing Python dependencies...")

# Önce requirements
req_file = "requirements.frozen.txt"
if not Path(req_file).exists():
    req_file = "requirements.txt"

if Path(req_file).exists():
    !pip install -q -r {req_file} 2>&1 | tail -5

# Colab GPU spesifik ek paketler
!pip install -q polars duckdb 2>&1 | tail -3

# Mevcut ortamı doğrula
import torch, numba, pyarrow, pandas
print(f"  torch:   {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"  numba:   {numba.__version__}")
print(f"  pyarrow: {pyarrow.__version__}")
print(f"  pandas:  {pandas.__version__}")
print("✅ Dependencies installed")


In [ ]:
# ═══════════════════════════════════════════════════════════
# Step 4: GPU Smoke Test
# ═══════════════════════════════════════════════════════════
from scripts.colab.colab_utils import smoke_test

results = smoke_test()
print("\n".join(f"  {k}: {v['status']}" for k, v in results.items()))
all_pass = all(v["status"] == "PASS" for v in results.values())
if all_pass:
    print("\n✅ ALL SMOKE TESTS PASS — GPU ready for training")
else:
    print("\n❌ Some smoke tests failed — check above")


In [ ]:
# ═══════════════════════════════════════════════════════════
# Step 5: Download Market Data (opsiyonel)
# ═══════════════════════════════════════════════════════════
# Colab'da Binance'ten direkt indirme, local data lake'e gerek yok

SYMBOLS = "BTCUSDT,ETHUSDT,SOLUSDT,BNBUSDT"

print(f"Downloading data for: {SYMBOLS}")
!python3 scripts/download_binance.py --symbols {SYMBOLS} 2>&1 | tail -10
print("\n✅ Data download complete")


In [ ]:
# ═══════════════════════════════════════════════════════════
# Step 6: Run Simulation (Leverage Baseline)
# ═══════════════════════════════════════════════════════════
# Mevcut simulation engine ile baz sonuçları üret

import sys
sys.path.insert(0, ".")
sys.path.insert(0, "alphaforge/src")

from simulation.authority import label_via_simulation, generate_labels_bulk
import numpy as np

MODE = "SCALP"
print(f"Running simulation in mode: {MODE}")

# Kısa smoke: simulate 1000 bar
!PYTHONPATH=alphaforge/src:. python3 -c "
from simulation.authority import generate_labels_bulk
import numpy as np
n = 2000
rng = np.random.default_rng(42)
closes = 100 + np.cumsum(rng.normal(0, 0.5, n))
highs = closes * (1 + rng.uniform(0, 0.01, n))
lows = closes * (1 - rng.uniform(0, 0.01, n))
labels = generate_labels_bulk(closes, highs, lows, 'SCALP')
print(f'Generating {len(labels[0])} labels... OK')
print(f'Action distribution: {np.bincount(labels[0])}')
" 2>&1

print("✅ Simulation complete")


In [ ]:
# ═══════════════════════════════════════════════════════════
# Step 7: AlphaForge Training
# ═══════════════════════════════════════════════════════════
# GPU baseline training

MODE = "SCALP"
SYMBOLS = "BTCUSDT,ETHUSDT,SOLUSDT,BNBUSDT"

print(f"Training: mode={MODE}, symbols={SYMBOLS}")

!PYTHONPATH=alphaforge/src:. python3 -m alphaforge.train \
    --mode {MODE} \
    --symbols {SYMBOLS} \
    --folds 6 \
    --output data/reports/colab-train-{MODE}.json 2>&1 | tail -20

print("\n✅ Training pipeline complete")


In [ ]:
# ═══════════════════════════════════════════════════════════
# Step 8: Sonuçları Google Drive'a Kaydet (opsiyonel)
# ═══════════════════════════════════════════════════════════
# from google.colab import drive
# drive.mount('/content/drive')
#
# DEST = f"/content/drive/MyDrive/af-results/{MODE}"
# !mkdir -p {DEST}
# !cp -r data/reports/ {DEST}/reports/
# !cp -r data_lake/ {DEST}/data_lake/
# print(f"Results saved to {DEST}")

# Local save (Colab VM'de kalır, session silinince gider)
!ls -la data/reports/colab-train-{MODE}.json 2>/dev/null && echo "✅ Results saved locally"


In [ ]:
# ═══════════════════════════════════════════════════════════
# Pipeline Summary
# ═══════════════════════════════════════════════════════════
print("=" * 50)
print("  ALPHAFORGE LEVERAGE-AWARE TRAINING — COLAB")
print("=" * 50)
print()
print("  Steps completed:")
print("  1. GPU check")
print("  2. Repo clone")
print("  3. Dependencies")
print("  4. Smoke test")
print("  5. Data download")
print("  6. Simulation")
print("  7. Training")
print()
print("  Sonuçlar: data/reports/ altında")
print("  Google Drive'a kaydetmek için Step 8'i aktifleştir")
print()
print("=" * 50)
